# Kidney dataset selection — audit (`_repaired`)

**v3 — poprawka błędu z v2.** W v2 użyłem `condition="Wildtype"` (wąski, historyczny filtr kidney) zamiast `condition=["Wildtype", "Wtype", "N/A"]`, którego użyłem dla brain. To był błąd — dawało to tylko 60 kandydatów zamiast 143, bo pomijało wszystkie datasety z `condition="N/A"` (submitter nie zadeklarował warunku — to nie znaczy, że próbka nie jest wildtype, patrz dyskusja w `brain_dataset.ipynb`). Ten notebook używa teraz dokładnie tego samego filtra biologicznego co brain, konsekwentnie w zapytaniu szerokim i finalnym. **Jedyne, co zostaje niezmienione względem historycznego kidney: zakres `mz_min=200, mz_max=900`** — to osobna oś, nie ruszam jej tutaj.

Reszta metodyki bez zmian: `DatasetExplorer.review_current()`/`.apply_review()` z `msi_dataset_manager.exploration.dataset_review` (zero zdublowanej logiki), zero zmian w bibliotece, zero nowych pobrań surowych danych.

In [4]:
import os
from pathlib import Path

current_path = Path.cwd().resolve()
repository_root = next(
    path
    for path in (current_path, *current_path.parents)
    if (path / "pyproject.toml").is_file()
)
os.chdir(repository_root)

repository_root

PosixPath('/home/max/repositories/MSIAutoEncoderWrapper')

In [5]:
import json

import pandas as pd
from IPython.display import display

from msi_dataset_manager.exploration import DatasetExplorer, DatasetReviewProfile

# REMARK: date i download DB is 12.08.2026 (DD, MM, YYYY) -- same cache as brain_dataset.ipynb.
explorer = DatasetExplorer(
    source="metaspace",
    cache_dir="assets/local/datasets/metaspace",
    refresh_cache=False,
)

## 1. Szeroka pula kandydatów — dokładnie filtr biologiczny z `brain_dataset.ipynb`

`condition=["Wildtype", "Wtype", "N/A"]` (nie samo `"Wildtype"`, patrz poprawka wyżej). To podnosi pulę z 60 (v2, błędne) do 143 kandydatów.

In [6]:
broad_filters = {
    "organism": "Mouse",
    "organism_part": "Kidney",
    "condition": ["Wildtype", "Wtype", "N/A"],
    "polarity": "Negative",
    "annotation_fdr": 0.1,
    "min_annotation_count": 1,
}
results = explorer.filter(broad_filters)
print(f"Found {len(results)} datasets")
display(results[["dataset_id", "name", "condition", "analyzer_type", "mz_min", "mz_max", "pixel_count"]])

METASPACE discovery:   0%|          | 0/3 [00:00<?, ?stage/s]

Current operation:   0%|          | 0/1 [00:00<?, ?operation/s]

Found 143 datasets


,dataset_id,name,condition,analyzer_type,mz_min,mz_max,pixel_count
0,2026-08-06_20h03m51s,dbdb-1770_2_S2_SM_Neg_20260731_AQ_ML,N/A,Orbitrap,70.001587,999.922974,52679
1,2026-08-06_20h04m41s,dbdb-1773_2_S2_SM_Neg_20260731_AQ_ML,N/A,Orbitrap,70.001732,999.683533,45501
2,2026-08-06_20h02m36s,dbm-1760_2_S2_SM_Neg_20260731_AQ_ML,N/A,Orbitrap,70.001617,999.698975,46311
3,2026-08-03_04h14m21s,dbm-1760_2_S2_SM_Neg_20260731_AQ,N/A,Orbitrap,70.001617,999.698975,46311
4,2026-08-03_04h15m34s,dbdb-1770_2_S2_SM_Neg_20260731_AQ,N/A,Orbitrap,70.001587,999.922974,52679
...,...,...,...,...,...,...,...
138,2017-05-03_17h48m18s,04272017_M_26_4,N/A,Orbitrap,100.007393,1499.989746,704
139,2017-05-03_17h51m16s,04272017_M_30_1,N/A,Orbitrap,100.017494,1499.989868,783
140,2017-05-03_17h53m50s,04272017_M_35_1,N/A,Orbitrap,100.007469,1499.990234,1140
141,2017-05-03_17h55m34s,04272017_M_37_4,N/A,Orbitrap,100.003838,1499.990479,1224


## 2. Ładowanie dotychczasowej ręcznej selekcji i przeniesienie jej wykluczeń do sesji

`data/kidney_workspace/configs/datasets/kidney/filter.json`: 30 zaakceptowanych, 30 ręcznie wykluczonych — ale z **węższej** puli (60, tylko `Wildtype`). Wszystkie te 60 ID mieszczą się w dzisiejszej puli 143 (sprawdzone niżej), więc przeniesienie ich do sesji jest bezpieczne.

In [7]:
existing_filter = json.load(open("data/kidney_workspace/configs/datasets/kidney/filter.json"))
existing_selection = json.load(open("data/kidney_workspace/configs/datasets/kidney/selection.json"))
existing_excluded_ids = existing_filter.get("exclude_dataset_ids", [])
existing_selected_ids = set(existing_selection["dataset_ids"])
print(f"existing selection: {len(existing_selected_ids)} selected, {len(existing_excluded_ids)} manually excluded")

known_ids = set(results["dataset_id"].astype(str))
missing = (set(existing_excluded_ids) | existing_selected_ids) - known_ids
assert not missing, f"existing IDs missing from the new broad pool: {missing}"

explorer.exclude(existing_excluded_ids)

existing selection: 30 selected, 30 manually excluded


,dataset_id,name,source,project_accession,project_url,organisms,organism_parts,condition,growth_conditions,diseases,...,unannotated_pixel_count,annotated_pixel_fraction,annotation_fdr,spatial_annotation_count,spatial_annotation_database_count,spatial_stats_status,molecule_count,unique_molecule_count,unique_molecules,excluded
0,2026-08-06_20h03m51s,dbdb-1770_2_S2_SM_Neg_20260731_AQ_ML,metaspace,None,https://metaspace2020.eu/dataset/2026-08-06_20...,Mus musculus (mouse),Kidney,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
1,2026-08-06_20h04m41s,dbdb-1773_2_S2_SM_Neg_20260731_AQ_ML,metaspace,None,https://metaspace2020.eu/dataset/2026-08-06_20...,Mus musculus (mouse),Kidney,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
2,2026-08-06_20h02m36s,dbm-1760_2_S2_SM_Neg_20260731_AQ_ML,metaspace,None,https://metaspace2020.eu/dataset/2026-08-06_20...,Mus musculus (mouse),Kidney,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
3,2026-08-03_04h14m21s,dbm-1760_2_S2_SM_Neg_20260731_AQ,metaspace,None,https://metaspace2020.eu/dataset/2026-08-03_04...,Mus musculus (mouse),Kidney,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
4,2026-08-03_04h15m34s,dbdb-1770_2_S2_SM_Neg_20260731_AQ,metaspace,None,https://metaspace2020.eu/dataset/2026-08-03_04...,Mus musculus (mouse),Kidney,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108,2017-05-03_17h48m18s,04272017_M_26_4,metaspace,None,https://metaspace2020.eu/dataset/2017-05-03_17...,Mus musculus (mouse),Kidney,N/A,N/A,,...,None,None,0.1,None,None,None,None,None,,False
109,2017-05-03_17h51m16s,04272017_M_30_1,metaspace,None,https://metaspace2020.eu/dataset/2017-05-03_17...,Mus musculus (mouse),Kidney,N/A,N/A,,...,None,None,0.1,None,None,None,None,None,,False
110,2017-05-03_17h53m50s,04272017_M_35_1,metaspace,None,https://metaspace2020.eu/dataset/2017-05-03_17...,Mus musculus (mouse),Kidney,N/A,N/A,,...,None,None,0.1,None,None,None,None,None,,False
111,2017-05-03_17h55m34s,04272017_M_37_4,metaspace,None,https://metaspace2020.eu/dataset/2017-05-03_17...,Mus musculus (mouse),Kidney,N/A,N/A,,...,None,None,0.1,None,None,None,None,None,,False


## 3. Ręczna kontrola jakości, której żadna reguła biblioteczna nie łapie

Poszerzenie puli o `N/A` wciągnęło też datasety, które żadna z reguł w `dataset_review.py` nie jest w stanie wykryć — to nie duplikaty, nie warianty kalibracyjne, nie fragmenty mikroanatomiczne. Przeszukałem nazwy pod kątem niedopasowania gatunku/tkanki i jawnych oznaczeń testowych:

- **`zebrafish_Brain_no wash_20211229_neg Analyte 4_1`** — metadane mówią `organism=Mouse`, `organism_part=Kidney`, ale nazwa jawnie mówi "zebrafish_Brain". To niespójność gatunku/tkanki w samej nazwie — nie ufam, że to faktycznie mysia nerka, więc wykluczam ręcznie.
- **`04272017_M_7_4 (TEST)`** — nazwa ma dosłowne `(TEST)`. Wykluczam ręcznie.

Sprawdziłem też pod kątem `drosophila`/`rat`/`human`/`calib`/`standard` — nic więcej się nie znalazło w tej puli.

In [8]:
suspect_pattern = r"zebrafish|drosophila|\brat\b|\bhuman\b|\btest\b|\(test\)|calib|standard"
suspects = results[results["name"].str.contains(suspect_pattern, case=False, na=False, regex=True)]
display(suspects[["dataset_id", "name", "condition", "organisms", "pixel_count"]])

EXPLICIT_QUALITY_EXCLUSIONS = {
    "2021-12-30_03h22m53s": "name is 'zebrafish_Brain_...' despite organism=Mouse/organism_part=Kidney metadata",
    "2017-05-02_17h04m13s": "name explicitly marked '(TEST)'",
}
explorer.exclude(list(EXPLICIT_QUALITY_EXCLUSIONS))

,dataset_id,name,condition,organisms,pixel_count
120,2021-12-30_03h22m53s,zebrafish_Brain_no wash_20211229_neg Analyte 4_1,N/A,Mouse,2966
136,2017-05-02_17h04m13s,04272017_M_7_4 (TEST),N/A,Mus musculus (mouse),609


,dataset_id,name,source,project_accession,project_url,organisms,organism_parts,condition,growth_conditions,diseases,...,unannotated_pixel_count,annotated_pixel_fraction,annotation_fdr,spatial_annotation_count,spatial_annotation_database_count,spatial_stats_status,molecule_count,unique_molecule_count,unique_molecules,excluded
0,2026-08-06_20h03m51s,dbdb-1770_2_S2_SM_Neg_20260731_AQ_ML,metaspace,None,https://metaspace2020.eu/dataset/2026-08-06_20...,Mus musculus (mouse),Kidney,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
1,2026-08-06_20h04m41s,dbdb-1773_2_S2_SM_Neg_20260731_AQ_ML,metaspace,None,https://metaspace2020.eu/dataset/2026-08-06_20...,Mus musculus (mouse),Kidney,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
2,2026-08-06_20h02m36s,dbm-1760_2_S2_SM_Neg_20260731_AQ_ML,metaspace,None,https://metaspace2020.eu/dataset/2026-08-06_20...,Mus musculus (mouse),Kidney,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
3,2026-08-03_04h14m21s,dbm-1760_2_S2_SM_Neg_20260731_AQ,metaspace,None,https://metaspace2020.eu/dataset/2026-08-03_04...,Mus musculus (mouse),Kidney,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
4,2026-08-03_04h15m34s,dbdb-1770_2_S2_SM_Neg_20260731_AQ,metaspace,None,https://metaspace2020.eu/dataset/2026-08-03_04...,Mus musculus (mouse),Kidney,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,2017-05-03_17h48m18s,04272017_M_26_4,metaspace,None,https://metaspace2020.eu/dataset/2017-05-03_17...,Mus musculus (mouse),Kidney,N/A,N/A,,...,None,None,0.1,None,None,None,None,None,,False
107,2017-05-03_17h51m16s,04272017_M_30_1,metaspace,None,https://metaspace2020.eu/dataset/2017-05-03_17...,Mus musculus (mouse),Kidney,N/A,N/A,,...,None,None,0.1,None,None,None,None,None,,False
108,2017-05-03_17h53m50s,04272017_M_35_1,metaspace,None,https://metaspace2020.eu/dataset/2017-05-03_17...,Mus musculus (mouse),Kidney,N/A,N/A,,...,None,None,0.1,None,None,None,None,None,,False
109,2017-05-03_17h55m34s,04272017_M_37_4,metaspace,None,https://metaspace2020.eu/dataset/2017-05-03_17...,Mus musculus (mouse),Kidney,N/A,N/A,,...,None,None,0.1,None,None,None,None,None,,False


## 4. Przegląd biblioteczny (`DatasetExplorer.review_current`)

Kidney wciąż nie ma wbudowanego profilu w `_PROFILES` (tylko `brain`/`liver`) — przekazuję równoważny `DatasetReviewProfile` z poziomu notebooka, tak jak w v2. **Zmiana względem v2:** `low_pixel_threshold=None` był oparty na węższej puli 60 (`Wildtype` only), gdzie minimum wynosiło 5025 pikseli. Pula 143 (z `N/A`) ma minimum **476** pikseli — próg musi się zmienić. Ustawiam `1000`, konserwatywnie: łapie tylko wyraźne przypadki poniżej jakiejkolwiek sensownej rozdzielczości tkankowej, nie ucina normalnej skali kidney (mediana tej puli to ~27 750 pikseli).

In [10]:
kidney_profile = DatasetReviewProfile(
    low_pixel_threshold=1000,  # v2 used None, calibrated to the narrower 60-record Wildtype-only pool; see markdown
    morphology_pattern=r"(?:cortex|medulla|papilla|pelvis|calyx|glomerul)",
    explicit_regional_names=frozenset(),
)
review = explorer.review_current(profile=kidney_profile)

print("available rules:", review.available_rules)
display(review.summary())

display(
    review.table.loc[
        review.table["duplicate_cluster_size"] > 1,
        ["duplicate_cluster_id", "dataset_id", "name", "pixel_count", "duplicate_confidence", "duplicate_excluded", "recommended_keeper_dataset_id"],
    ].sort_values(["duplicate_confidence", "duplicate_cluster_id"])
)
display(
    review.table.loc[
        review.table["mz_shift_qc_variant"] | review.table["morphology_hint"].eq("regional_or_microregion") | review.table["low_pixel_flag"],
        ["dataset_id", "name", "pixel_count", "mz_shift_qc_variant", "morphology_hint", "low_pixel_flag"],
    ]
)

available rules: ('high_confidence_duplicates', 'mz_shift_qc_variants', 'explicit_regional_fragments')


,rule,dataset_count
0,high_confidence_duplicates,9
1,mz_shift_qc_variants,1
2,explicit_regional_fragments,0


,duplicate_cluster_id,dataset_id,name,pixel_count,duplicate_confidence,duplicate_excluded,recommended_keeper_dataset_id
40,technical-0077,2026-04-21_18h15m40s,Kidney_3_S1_SM_Neg_20260421_AQ_1.95A,45836,ambiguous_shared_template,False,<NA>
42,technical-0077,2026-04-21_18h09m29s,Kidney_1_3_S1_SM_Neg_20260421_AQ,45836,ambiguous_shared_template,False,<NA>
39,technical-0099,2026-04-21_18h16m54s,Kidney_2_S1_SM_Neg_20260421_AQ_1.95A,81071,ambiguous_shared_template,False,<NA>
44,technical-0099,2026-04-21_18h07m29s,Kidney_1_2_S1_SM_Neg_20260421_AQ,81071,ambiguous_shared_template,False,<NA>
10,technical-0067,2026-07-30_19h53m34s,CT179_1_S5_SM_Pos_20260713_AQ_ML,33464,high_confidence_duplicate,True,2026-07-30_19h52m06s
12,technical-0067,2026-07-30_19h52m06s,CT179_1_S5_SM_Pos_20260713_AQ,33464,high_confidence_duplicate,False,2026-07-30_19h52m06s
1,technical-0076,2026-08-06_20h04m41s,dbdb-1773_2_S2_SM_Neg_20260731_AQ_ML,45501,high_confidence_duplicate,True,2026-08-03_04h16m56s
6,technical-0076,2026-08-03_04h16m56s,dbdb-1773_2_S2_SM_Neg_20260731_AQ,45501,high_confidence_duplicate,False,2026-08-03_04h16m56s
2,technical-0078,2026-08-06_20h02m36s,dbm-1760_2_S2_SM_Neg_20260731_AQ_ML,46311,high_confidence_duplicate,True,2026-08-03_04h14m21s
3,technical-0078,2026-08-03_04h14m21s,dbm-1760_2_S2_SM_Neg_20260731_AQ,46311,high_confidence_duplicate,False,2026-08-03_04h14m21s


,dataset_id,name,pixel_count,mz_shift_qc_variant,morphology_hint,low_pixel_flag
37,2026-05-05_00h26m37s,Kidney_3_2_S2_SM_Neg_20260420_AQ_1.95A_Seq2,939,False,whole_section_likely,True
38,2026-04-22_21h03m00s,kidney_test_metabolites_null_mz_shift_10_til_550,8575,True,whole_section_likely,False
43,2026-04-21_18h13m27s,Kidney_4_2_S1_SM_Neg_20260421_AQ_1.95A,805,False,whole_section_likely,True
105,2017-05-03_17h41m47s,04272017_M_24_4,476,False,whole_section_likely,True
106,2017-05-03_17h48m18s,04272017_M_26_4,704,False,whole_section_likely,True
107,2017-05-03_17h51m16s,04272017_M_30_1,783,False,whole_section_likely,True


## 5. Zastosowanie reguł i finalna, poprawiona lista

`high_confidence_duplicates` (9 — głównie pary `*_AQ`/`*_AQ_ML`, ten sam wzorzec co brain/liver) + `mz_shift_qc_variants` (1) + `explicit_regional_fragments` (0, puste `explicit_regional_names`). `ambiguous_shared_template` (2 pary `Kidney_N_S1_SM_Neg_...`, prawdopodobnie różne zwierzęta na tym samym szablonie akwizycji — patrz przypadek DKFZACLY w `liver_dataset_repaired.ipynb`) **nie** jest wykluczane. `morphology_hint`/`low_pixel_flag` zostają tylko doradczo w tabeli.

In [11]:
applied_rules = ["high_confidence_duplicates", "mz_shift_qc_variants", "explicit_regional_fragments"]
explorer.apply_review(review, rules=applied_rules)

final_filters = {
    "organism": "Mouse",
    "organism_part": "Kidney",
    "condition": ["Wildtype", "Wtype", "N/A"],
    "polarity": "Negative",
    "mz_min": 200,
    "mz_max": 900,
    "annotation_fdr": 0.1,
    "min_annotation_count": 1,
    "include_molecule_stats": True,
    "include_spatial_annotation_stats": False,  # see brain_dataset.ipynb section 6 for the cost rationale
}
# exclude_dataset_ids intentionally omitted -- session-level exclusions from steps 2/3/5 persist across this re-query.
results_kidney_repaired = explorer.filter(final_filters)
print(f"repaired kidney shortlist: {len(results_kidney_repaired)} datasets (previously {len(existing_selected_ids)})")
display(results_kidney_repaired[["dataset_id", "name", "analyzer_type", "pixel_count", "molecule_count", "unique_molecule_count"]])

METASPACE discovery:   0%|          | 0/3 [00:00<?, ?stage/s]

Current operation:   0%|          | 0/1 [00:00<?, ?operation/s]

repaired kidney shortlist: 79 datasets (previously 30)


,dataset_id,name,analyzer_type,pixel_count,molecule_count,unique_molecule_count
0,2026-08-03_04h14m21s,dbm-1760_2_S2_SM_Neg_20260731_AQ,Orbitrap,46311,50,5
1,2026-08-03_04h15m34s,dbdb-1770_2_S2_SM_Neg_20260731_AQ,Orbitrap,52679,38,2
2,2026-08-03_04h13m27s,dbm-1758_4_S2_SM_Neg_20260731_AQ,Orbitrap,56701,55,13
3,2026-08-03_04h16m56s,dbdb-1773_2_S2_SM_Neg_20260731_AQ,Orbitrap,45501,23,2
4,2026-07-30_20h05m30s,CT198_2_S4-2_SM_Pos_20260707_AQ_ML,Orbitrap,33174,9,5
...,...,...,...,...,...,...
74,2017-05-03_17h41m47s,04272017_M_24_4,Orbitrap,476,198,10
75,2017-05-03_17h48m18s,04272017_M_26_4,Orbitrap,704,216,13
76,2017-05-03_17h51m16s,04272017_M_30_1,Orbitrap,783,186,7
77,2017-05-03_17h53m50s,04272017_M_35_1,Orbitrap,1140,167,6


## 6. Które datasety zostały wycięte / dodane względem dotychczasowego `kidney/` — pełne porównanie

Poszerzenie `condition` do `N/A` może zarówno **dodawać** kandydatów (spełniają teraz filtr, których wcześniej filtr w ogóle nie widział), jak i **wycinać** (duplikaty/QC/jakość). Tabela rozróżnia oba przypadki.

In [12]:
repaired_selected_ids = set(results_kidney_repaired["dataset_id"].astype(str))
review_reasons = {
    dataset_id: "high_confidence_duplicate"
    for dataset_id in review.exclusion_ids(["high_confidence_duplicates"])
}
review_reasons.update({
    dataset_id: "mz_shift_qc_variant"
    for dataset_id in review.exclusion_ids(["mz_shift_qc_variants"])
})
review_reasons.update({
    dataset_id: f"manual_quality_exclusion ({reason})"
    for dataset_id, reason in EXPLICIT_QUALITY_EXCLUSIONS.items()
})

comparison = results[["dataset_id", "name", "condition"]].copy()
comparison["in_original_selection"] = comparison["dataset_id"].isin(existing_selected_ids)
comparison["in_repaired_selection"] = comparison["dataset_id"].isin(repaired_selected_ids)
comparison["status"] = "unchanged_excluded"
comparison.loc[comparison["in_original_selection"] & comparison["in_repaired_selection"], "status"] = "unchanged_included"
comparison.loc[comparison["in_original_selection"] & ~comparison["in_repaired_selection"], "status"] = "REMOVED"
comparison.loc[~comparison["in_original_selection"] & comparison["in_repaired_selection"], "status"] = "ADDED"
comparison["reason"] = comparison["dataset_id"].map(review_reasons).fillna("")

changed = comparison.loc[comparison["status"].isin(["REMOVED", "ADDED"])].sort_values("status")
print(f"datasets whose accept/exclude status changed: {len(changed)}")
display(changed)

print(comparison["status"].value_counts())
print(f"\ntotal: {comparison['in_original_selection'].sum()} (original) -> {comparison['in_repaired_selection'].sum()} (repaired)")

datasets whose accept/exclude status changed: 51


,dataset_id,name,condition,in_original_selection,in_repaired_selection,status,reason
3,2026-08-03_04h14m21s,dbm-1760_2_S2_SM_Neg_20260731_AQ,N/A,False,True,ADDED,
36,2026-05-05_00h26m58s,Kidney_3_1_S2_SM_Neg_20260420_AQ_1.95A_Seq1,N/A,False,True,ADDED,
37,2026-05-05_00h26m37s,Kidney_3_2_S2_SM_Neg_20260420_AQ_1.95A_Seq2,N/A,False,True,ADDED,
39,2026-04-21_18h16m54s,Kidney_2_S1_SM_Neg_20260421_AQ_1.95A,N/A,False,True,ADDED,
40,2026-04-21_18h15m40s,Kidney_3_S1_SM_Neg_20260421_AQ_1.95A,N/A,False,True,ADDED,
41,2026-04-21_18h13m44s,Kidney_4_1_S1_SM_Neg_20260421_AQ_1.95A,N/A,False,True,ADDED,
42,2026-04-21_18h09m29s,Kidney_1_3_S1_SM_Neg_20260421_AQ,N/A,False,True,ADDED,
43,2026-04-21_18h13m27s,Kidney_4_2_S1_SM_Neg_20260421_AQ_1.95A,N/A,False,True,ADDED,
44,2026-04-21_18h07m29s,Kidney_1_2_S1_SM_Neg_20260421_AQ,N/A,False,True,ADDED,
45,2025-10-30_20h03m38s,C163_1_S2_SM_Neg_20251024_SZ,N/A,False,True,ADDED,


status
unchanged_excluded    63
ADDED                 50
unchanged_included    29
REMOVED                1
Name: count, dtype: int64

total: 30 (original) -> 79 (repaired)


In [9]:
output_path = Path("data/kidney_workspace/configs/datasets/kidney_repaired")
exported = explorer.export_selection(output_path, sort_by="download_size_bytes", ascending=False)
exported

{'filters': PosixPath('data/kidney_workspace/configs/datasets/kidney_repaired/filter.json'),
 'selection': PosixPath('data/kidney_workspace/configs/datasets/kidney_repaired/selection.json')}

## Podsumowanie

- **Poprawka v2→v3:** filtr biologiczny ujednolicony z brain (`condition=["Wildtype","Wtype","N/A"]`) — pula kandydatów 60 → 143. `mz_min=200, mz_max=900` bez zmian (osobna oś, nierozwiązywana tu).
- 2 datasety wykluczone ręcznie z powodów, których żadna reguła biblioteczna nie łapie: niedopasowanie gatunku/tkanki w nazwie (`zebrafish_Brain`) i jawne oznaczenie `(TEST)`.
- Przegląd biblioteczny: 9 `high_confidence_duplicates`, 1 `mz_shift_qc_variant`, 2 pary `ambiguous_shared_template` (niewykluczone).
- `low_pixel_threshold` przeliczony na `1000` dla tej szerszej puli (v2 miał błędnie `None`, oparte na węższej puli).
- Wynik: **30 → 79** (patrz pełna tabela zmian w sekcji 6 — to głównie nowo dopuszczone datasety `condition=N/A`, nie usunięcia).
- Eksport do `data/kidney_workspace/configs/datasets/kidney_repaired/` — istniejący `kidney/` nie został nadpisany.